In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/harunshimanto/epileptic-seizure-recognition/Epileptic Seizure Recognition.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("/kaggle/input/datasets/harunshimanto/epileptic-seizure-recognition/Epileptic Seizure Recognition.csv")
print(df.head())

      Unnamed   X1   X2   X3   X4   X5   X6   X7   X8   X9  ...  X170  X171  \
0  X21.V1.791  135  190  229  223  192  125   55   -9  -33  ...   -17   -15   
1  X15.V1.924  386  382  356  331  320  315  307  272  244  ...   164   150   
2     X8.V1.1  -32  -39  -47  -37  -32  -36  -57  -73  -85  ...    57    64   
3   X16.V1.60 -105 -101  -96  -92  -89  -95 -102 -100  -87  ...   -82   -81   
4   X20.V1.54   -9  -65  -98 -102  -78  -48  -16    0  -21  ...     4     2   

   X172  X173  X174  X175  X176  X177  X178  y  
0   -31   -77  -103  -127  -116   -83   -51  4  
1   146   152   157   156   154   143   129  1  
2    48    19   -12   -30   -35   -35   -36  5  
3   -80   -77   -85   -77   -72   -69   -65  5  
4   -12   -32   -41   -65   -83   -89   -73  5  

[5 rows x 180 columns]


In [4]:
df = df.drop(columns=["Unnamed"], errors="ignore")
df["y"] = (df["y"] == 1).astype(int)

In [5]:
X = df.drop("y", axis=1)
y = df["y"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [8]:
l1_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=1.0,
    max_iter=3000
)

In [9]:
l1_model.fit(X_train, y_train)

LogisticRegression(max_iter=3000, penalty='l1', solver='liblinear')

In [10]:
l1_train_pred = l1_model.predict(X_train)
l1_test_pred = l1_model.predict(X_test)

In [11]:
# RIDGE
l2_model = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    C=1.0,
    max_iter=3000
)

In [12]:
l2_model.fit(X_train, y_train)

LogisticRegression(max_iter=3000)

In [13]:
l2_train_pred = l2_model.predict(X_train)
l2_test_pred = l2_model.predict(X_test)

In [14]:
#elastic net
elastic_model = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=0.5,
    C=1.0,
    max_iter=5000
)

In [15]:
elastic_model.fit(X_train, y_train)

LogisticRegression(l1_ratio=0.5, max_iter=5000, penalty='elasticnet',
                   solver='saga')

In [16]:
en_train_pred = elastic_model.predict(X_train)
en_test_pred = elastic_model.predict(X_test)

In [17]:
print("===== L1 (LASSO) =====")
print("Train Accuracy:", accuracy_score(y_train, l1_train_pred))
print("Test Accuracy:", accuracy_score(y_test, l1_test_pred))

print("\n===== L2 (RIDGE) =====")
print("Train Accuracy:", accuracy_score(y_train, l2_train_pred))
print("Test Accuracy:", accuracy_score(y_test, l2_test_pred))

print("\n===== ELASTIC NET =====")
print("Train Accuracy:", accuracy_score(y_train, en_train_pred))
print("Test Accuracy:", accuracy_score(y_test, en_test_pred))

===== L1 (LASSO) =====
Train Accuracy: 0.8270652173913043
Test Accuracy: 0.8152173913043478

===== L2 (RIDGE) =====
Train Accuracy: 0.8283695652173914
Test Accuracy: 0.8160869565217391

===== ELASTIC NET =====
Train Accuracy: 0.8277173913043478
Test Accuracy: 0.8147826086956522


In [18]:
coef = l1_model.coef_[0]

non_zero = np.sum(coef != 0)
total = len(coef)

print("\n===== SPARSITY ANALYSIS =====")
print("Non-zero features:", non_zero)
print("Total features:", total)
print("Sparsity (%):", (non_zero / total) * 100)


===== SPARSITY ANALYSIS =====
Non-zero features: 130
Total features: 178
Sparsity (%): 73.03370786516854
